In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps as UTC datetimes
metric_df = pd.read_csv("metric.csv")
log_df = pd.read_csv("log.csv")
trace_df = pd.read_csv("trace.csv")
error_df = pd.read_csv("error_logs.csv")

metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Ensure numeric value columns (coerce non-numeric to NaN)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Helper to compute aggregated statistics for a grouped numeric series
def compute_stats(df, group_cols, value_col, top_n=20):
    grp = df.groupby(group_cols)[value_col]
    agg_base = grp.agg(count='size', min='min', max='max', mean='mean')
    q50 = grp.quantile(0.5).rename('p50')
    q90 = grp.quantile(0.9).rename('p90')
    q95 = grp.quantile(0.95).rename('p95')
    combined = agg_base.join(q50).join(q90).join(q95).reset_index()
    # Order columns: group cols..., count, min, p50, p90, p95, max, mean
    col_order = list(group_cols) + ['count', 'min', 'p50', 'p90', 'p95', 'max', 'mean']
    combined = combined[col_order]
    combined = combined.sort_values('count', ascending=False).head(top_n)
    # Round numeric columns for compactness
    for c in ['min','p50','p90','p95','max','mean']:
        if c in combined.columns:
            combined[c] = combined[c].round(6)
    return combined

# 1) metric.csv summary: (cmdb_id, kpi_name)
metric_summary = compute_stats(metric_df, ['cmdb_id','kpi_name'], 'value', top_n=20)

# 2) log.csv summary: (cmdb_id, log_name)
log_summary = compute_stats(log_df, ['cmdb_id','log_name'], 'value', top_n=20)

# 3) trace.csv summary: (cmdb_id, trace_name)
trace_summary = compute_stats(trace_df, ['cmdb_id','trace_name'], 'value', top_n=20)

# 4) error_logs.csv summary
error_total_rows = len(error_df)
cmdb_counts = error_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
error_earliest_overall = error_df['timestamp'].min()
error_latest_overall = error_df['timestamp'].max()
per_cmdb = error_df.groupby('cmdb_id')['timestamp'].agg(earliest_utc='min', latest_utc='max').reset_index()
per_cmdb = per_cmdb.merge(cmdb_counts, on='cmdb_id').sort_values('count', ascending=False).head(20)
error_overall = pd.DataFrame({
    'total_rows':[error_total_rows],
    'distinct_cmdb_count':[cmdb_counts.shape[0]],
    'earliest_utc':[error_earliest_overall],
    'latest_utc':[error_latest_overall]
})

# Return compact summaries (IPython will display these variables)
metric_summary, log_summary, trace_summary, error_overall, per_cmdb

```
Out[1]:
```
summary_text = (
    "Summary of telemetry (plain English):\n\n"
    "1) Metrics (top results):\n"
    "- The 'carts' service has 25 metric points showing high CPU (mean ~38.5, max ~79.3), "
    "large disk I/O variability (mean ~4.97k, max ~66.3k) and high memory usage (mean ~4.93e8). "
    "Sockets and latency KPIs for 'carts' are also present.\n"
    "- The 'carts-db' component (25 points) shows very high disk I/O (mean ~1.99e6, max ~2.69e6) "
    "and moderate CPU/memory values.\n"
    "- 'catalogue' and 'catalogue-db' also appear in top metric pairs but with much smaller CPU and latency values compared to 'carts' and 'carts-db'.\n\n"
    "2) Logs (top results):\n"
    "- log.error_count is 0 across listed services (multiple components have 25 points with zero errors).\n"
    "- log.row_count shows traffic/row counts per service (e.g., front-end mean ~1898, user mean ~572.6, catalogue mean ~156.8).\n\n"
    "3) Traces:\n"
    "- No trace metrics were present in the trace.csv summary (empty result).\n\n"
    "4) Error logs:\n"
    "- error_logs.csv is empty (total rows = 0), so there are no raw error log entries to inspect.\n\n"
    "Overall conclusion:\n"
    "- There are no logged errors or error-traces in the provided data, but metric telemetry points to resource pressure on 'carts' (high CPU, disk I/O spikes, large memory) and very high disk I/O on 'carts-db'. "
    "These metric signals are the most likely areas to investigate further for performance or resource-related root causes."
)

summary_text


The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count           min           p50           p90           p95           max          mean
0          carts         cpu     25  1.285117e+00  1.454931e+01  7.897347e+01  7.914544e+01  7.929470e+01  3.849495e+01
1          carts      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  4.643662e+04  6.628389e+04  4.973187e+03
2          carts  latency-50     25  1.090400e-02  1.509700e-02  5.246000e-02  5.330100e-02  5.511200e-02  3.014000e-02
3          carts  latency-90     25  2.231700e-02  4.369400e-02  9.116300e-02  9.129000e-02  9.433700e-02  5.611600e-02
4          carts         mem     25  2.129547e+08  3.919882e+08  9.214567e+08  9.300768e+08  9.401712e+08  4.930647e+08
5          carts      socket     25  9.000000e+00  1.220000e+01  1.471667e+01  1.547667e+01  1.586667e+01  1.232324e+01
6          carts    workload     25  7.776617e+00  8.183383e+00  8.536883e+00  8.657317e+00  8.854533e+00  8.222053e+00
7       carts-db         cpu     25  2.535627e+00  2.899945e+00  3.544775e+00  3.633264e+00  4.009911e+00  3.038856e+00
8       carts-db      diskio     25  1.699663e+06  1.962126e+06  2.245855e+06  2.330160e+06  2.688884e+06  1.994608e+06
9       carts-db         mem     25  7.706442e+07  8.002929e+07  8.278957e+07  8.301793e+07  8.330559e+07  8.015200e+07
10      carts-db      socket     25  6.000000e+00  6.416667e+00  7.000000e+00  7.000000e+00  7.000000e+00  6.496667e+00
11     catalogue         cpu     25  1.595800e-01  1.848380e-01  2.049380e-01  2.081030e-01  2.131040e-01  1.879760e-01
12     catalogue  latency-50     25  3.005000e-03  3.070000e-03  3.303000e-03  3.307000e-03  3.317000e-03  3.142000e-03
13     catalogue  latency-90     25  4.610000e-03  4.726000e-03  6.249000e-03  6.282000e-03  6.448000e-03  5.160000e-03
14     catalogue         mem     25  5.997500e+06  6.073344e+06  6.105197e+06  6.109239e+06  6.114181e+06  6.068649e+06
15     catalogue      socket     25  7.000000e+00  7.000000e+00  7.000000e+00  7.000000e+00  7.000000e+00  7.000000e+00
16     catalogue    workload     25  3.953400e+00  4.123300e+00  4.292630e+00  4.364614e+00  4.403200e+00  4.124365e+00
17  catalogue-db         cpu     25  1.818660e-01  1.986200e-01  2.132160e-01  2.194690e-01  2.205860e-01  1.991970e-01
18  catalogue-db      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
19  catalogue-db         mem     25  4.306903e+08  4.306913e+08  4.306935e+08  4.306937e+08  4.306945e+08  4.306916e+08,          cmdb_id         log_name  count  min     p50     p90     p95   max         mean
4      catalogue  log.error_count     25    0     0.0     0.0     0.0     0     0.000000
5      catalogue    log.row_count     25   62   163.0   166.6   170.2   175   156.800000
9         orders    log.row_count     25   50   124.0   130.0   130.0   132   118.800000
8         orders  log.error_count     25    0     0.0     0.0     0.0     0     0.000000
7      front-end    log.row_count     25  809  1965.0  2063.8  2078.2  2088  1898.080000
6      front-end  log.error_count     25    0     0.0     0.0     0.0     0     0.000000
17      shipping    log.row_count     25   25    62.0    65.0    65.0    66    59.400000
16      shipping  log.error_count     25    0     0.0     0.0     0.0     0     0.000000
19          user    log.row_count     25  244   597.0   612.6   617.0   630   572.600000
18          user  log.error_count     25    0     0.0     0.0     0.0     0     0.000000
10       payment  log.error_count     25    0     0.0     0.0     0.0     0     0.000000
11       payment    log.row_count     25   41   102.0   105.0   105.0   106    97.800000
12  queue-master  log.error_count     25    0     0.0     0.0     0.0     0     0.000000
13  queue-master    log.row_count     25  225   558.0   585.0   585.0   594   534.600000
2       carts-db  log.error_count     24    0     0.0     0.0     0.0     0     0.000000
3       carts-db    log.row_count     24    1     1.0     1.0     1.0     2     1.041667
0          carts  log.error_count     24    0     0.0     0.0     0.0     0     0.000000
1          carts    log.row_count     24   25    75.0   100.0   100.0   125    60.458333
14    session-db  log.error_count      4    0     0.0     0.0     0.0     0     0.000000
15    session-db    log.row_count      4    5     5.0     5.0     5.0     5     5.000000, Empty DataFrame
Columns: [cmdb_id, trace_name, count, min, p50, p90, p95, max, mean]
Index: [],    total_rows  distinct_cmdb_count earliest_utc latest_utc
0           0                    0          NaT        NaT, Empty DataFrame
Columns: [cmdb_id, earliest_utc, latest_utc, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing metric_df and log_df if present; otherwise load from CSVs
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
    log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Ensure timestamps parsed (in case they were loaded previously without parsing)
if metric_df['timestamp'].dtype == 'int64' or metric_df['timestamp'].dtype == 'float64':
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
if log_df['timestamp'].dtype == 'int64' or log_df['timestamp'].dtype == 'float64':
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Target components and incident window (UTC)
targets = ['carts','catalogue','orders','payment','user']
start = pd.to_datetime("2024-01-19 16:40:24", utc=True)
end = pd.to_datetime("2024-01-19 17:10:24", utc=True)

# ---- Metric analysis ----
metric_sub = metric_df[metric_df['cmdb_id'].isin(targets)].copy()

# If no rows for these components, create empty result frames later
if metric_sub.empty:
    metric_anomalies = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_type','anomaly_count',
                                             'earliest_anomaly_utc','max_value_in_window','global_p95','global_p5'])
else:
    # Compute global P95 and P5 per (cmdb_id, kpi_name) BEFORE filtering by window
    q = metric_sub.groupby(['cmdb_id','kpi_name'])['value'].quantile
    p95 = q(0.95).rename('global_p95').reset_index()
    p5  = q(0.05).rename('global_p5').reset_index()
    thresholds = p95.merge(p5, on=['cmdb_id','kpi_name'])
    # Filter to incident window (inclusive)
    metric_win = metric_sub[(metric_sub['timestamp'] >= start) & (metric_sub['timestamp'] <= end)].copy()
    if metric_win.empty:
        metric_anomalies = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_type','anomaly_count',
                                                 'earliest_anomaly_utc','max_value_in_window','global_p95','global_p5'])
    else:
        # Join thresholds
        metric_win = metric_win.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
        # Determine anomaly type per row
        kw_high = ['cpu','mem','diskio','latency','socket']
        kw_low  = ['workload','row_count']
        def detect_anomaly(row):
            name = str(row['kpi_name']).lower()
            val = row['value']
            p95v = row['global_p95']
            p5v = row['global_p5']
            # high KPIs
            for k in kw_high:
                if k in name:
                    if pd.notna(val) and pd.notna(p95v) and val >= p95v:
                        return 'high'
                    else:
                        return None
            # low KPIs
            for k in kw_low:
                if k in name:
                    if pd.notna(val) and pd.notna(p5v) and val <= p5v:
                        return 'low'
                    else:
                        return None
            return None
        metric_win['anomaly_type'] = metric_win.apply(detect_anomaly, axis=1)
        anomalies = metric_win[metric_win['anomaly_type'].notna()].copy()
        if anomalies.empty:
            metric_anomalies = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_type','anomaly_count',
                                                     'earliest_anomaly_utc','max_value_in_window','global_p95','global_p5'])
        else:
            agg = anomalies.groupby(['cmdb_id','kpi_name','anomaly_type']).agg(
                anomaly_count=('value','size'),
                earliest_anomaly_utc=('timestamp','min'),
                max_value_in_window=('value','max'),
                global_p95=('global_p95','first'),
                global_p5=('global_p5','first')
            ).reset_index()
            # Round numeric columns for compactness
            agg['max_value_in_window'] = agg['max_value_in_window'].round(6)
            agg['global_p95'] = agg['global_p95'].round(6)
            agg['global_p5'] = agg['global_p5'].round(6)
            metric_anomalies = agg.sort_values('anomaly_count', ascending=False).head(20)

# ---- Log analysis ----
log_sub = log_df[log_df['cmdb_id'].isin(targets) & log_df['log_name'].isin(['log.row_count','log.error_count'])].copy()

if log_sub.empty:
    log_anomalies = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_type','anomaly_count',
                                          'earliest_anomaly_utc','max_value_in_window','global_p95','global_p5'])
else:
    # Compute global thresholds BEFORE filtering
    p95_log = log_sub.groupby(['cmdb_id','log_name'])['value'].quantile(0.95).rename('global_p95').reset_index()
    p5_log  = log_sub.groupby(['cmdb_id','log_name'])['value'].quantile(0.05).rename('global_p5').reset_index()
    thresholds_log = p95_log.merge(p5_log, on=['cmdb_id','log_name'])
    # Filter to window
    log_win = log_sub[(log_sub['timestamp'] >= start) & (log_sub['timestamp'] <= end)].copy()
    if log_win.empty:
        log_anomalies = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_type','anomaly_count',
                                              'earliest_anomaly_utc','max_value_in_window','global_p95','global_p5'])
    else:
        log_win = log_win.merge(thresholds_log, on=['cmdb_id','log_name'], how='left')
        # Detect anomalies:
        # - log.row_count <= global_p5  => low
        # - log.error_count > 0         => high
        def detect_log_anom(row):
            name = str(row['log_name']).lower()
            val = row['value']
            p5v = row.get('global_p5', pd.NA)
            if name == 'log.row_count':
                if pd.notna(val) and pd.notna(p5v) and val <= p5v:
                    return 'low'
            if name == 'log.error_count':
                # anomaly if any error count > 0
                if pd.notna(val) and val > 0:
                    return 'high'
            return None
        log_win['anomaly_type'] = log_win.apply(detect_log_anom, axis=1)
        log_anom_rows = log_win[log_win['anomaly_type'].notna()].copy()
        if log_anom_rows.empty:
            log_anomalies = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_type','anomaly_count',
                                                  'earliest_anomaly_utc','max_value_in_window','global_p95','global_p5'])
        else:
            agg_log = log_anom_rows.groupby(['cmdb_id','log_name','anomaly_type']).agg(
                anomaly_count=('value','size'),
                earliest_anomaly_utc=('timestamp','min'),
                max_value_in_window=('value','max'),
                global_p95=('global_p95','first'),
                global_p5=('global_p5','first')
            ).reset_index()
            agg_log['max_value_in_window'] = agg_log['max_value_in_window'].round(6)
            agg_log['global_p95'] = agg_log['global_p95'].round(6)
            agg_log['global_p5'] = agg_log['global_p5'].round(6)
            log_anomalies = agg_log.sort_values('anomaly_count', ascending=False).head(20)

# Display compact outputs (IPython will show these variables)
metric_anomalies, log_anomalies

```
Out[2]:
```
Summary of findings (incident window 2024-01-19 16:40:24 → 2024-01-19 17:10:24 UTC):

Key points
- Multiple services show metric anomalies within the window. The most pervasive single anomaly is "catalogue - socket" (25 anomalous points — earliest 2024-01-19 16:43 UTC), i.e., socket metric at or above its global P95.
- The carts service shows several resource-related high anomalies (cpu, diskio, mem, latency) and a low workload anomaly; these occur with small counts but are present across the window and have high maxima relative to their global thresholds.
- Several services (catalogue, orders, payment, user, carts) show low log.row_count anomalies (reduced throughput) during the window. No log.error_count high anomalies were reported.

Metric anomaly highlights (compact)
- catalogue / socket — type=high, anomaly_count=25, earliest=2024-01-19 16:43 UTC, max=7.0 (global_p95=7.0).
- payment / socket — type=high, anomaly_count=10, earliest=2024-01-19 16:58 UTC, max=5.0 (global_p95=5.0).
- user / socket — type=high, anomaly_count=5, earliest=2024-01-19 17:03 UTC, max=19.0 (global_p95=19.0).
- carts:
  - cpu — high, anomaly_count=2, earliest=2024-01-19 16:57 UTC, max≈79.29 (global_p95≈79.15).
  - diskio — high, anomaly_count=2, earliest=2024-01-19 16:43 UTC, max≈66283.89 (global_p95≈46436.62).
  - mem — high, anomaly_count=2, earliest=2024-01-19 16:56 UTC, max≈940,171,200 (global_p95≈930,076,800).
  - latency-50 / latency-90 — high (each with 2 anomalies) with earliest times ~16:56–17:04 UTC.
  - workload — low, anomaly_count=2, earliest=2024-01-19 17:04 UTC (value dropped below global P5).

Other services with metric high anomalies (counts = 2 each): catalogue cpu, catalogue mem, catalogue latency, orders cpu/latency/mem, payment cpu.

Log anomaly highlights (compact)
- carts / log.row_count — low, anomaly_count=8, earliest=2024-01-19 16:45 UTC, max_in_window=25, global_p5=25.0
- catalogue / log.row_count — low, anomaly_count=2, earliest=2024-01-19 16:43 UTC, max_in_window=105, global_p5=115.0
- orders / log.row_count — low, anomaly_count=2, earliest=2024-01-19 16:43 UTC, max_in_window=84, global_p5=89.6
- payment / log.row_count — low, anomaly_count=2, earliest=2024-01-19 16:43 UTC, max_in_window=66, global_p5=72.0
- user / log.row_count — low, anomaly_count=2, earliest=2024-01-19 16:43 UTC, max_in_window=395, global_p5=424.8

Interpretation / recommended next steps
- The data shows concurrent resource and socket anomalies plus reduced request/row counts. Two likely investigation angles:
  1) Socket-related saturation or connection issues affecting catalogue (and to a lesser extent payment and user) — investigate socket utilization, connection pool exhaustion, and network/connectivity for those services.
  2) Resource pressure on carts (high CPU, disk I/O, memory, and increased latency) coincident with lower workload — investigate node resource usage, throttling, I/O contention, and whether degraded performance reduced throughput.
- Prioritize checking socket metrics and connection counts for catalogue first (highest anomaly count), and inspect carts hosts for CPU/disk/memory spikes and related logs/traces around the earliest anomaly times (16:43–16:57 UTC).

If you want, I can produce per-timestamp anomaly samples or list the specific rows that were flagged for any single service.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name anomaly_type  anomaly_count      earliest_anomaly_utc  max_value_in_window    global_p95     global_p5
11  catalogue      socket         high             25 2024-01-19 16:43:00+00:00         7.000000e+00  7.000000e+00  7.000000e+00
23    payment      socket         high             10 2024-01-19 16:58:00+00:00         5.000000e+00  5.000000e+00  4.000000e+00
29       user      socket         high              5 2024-01-19 17:03:00+00:00         1.900000e+01  1.900000e+01  1.733333e+01
3       carts  latency-90         high              2 2024-01-19 16:56:00+00:00         9.433700e-02  9.129000e-02  2.236900e-02
4       carts         mem         high              2 2024-01-19 16:56:00+00:00         9.401712e+08  9.300768e+08  2.131318e+08
1       carts      diskio         high              2 2024-01-19 16:43:00+00:00         6.628389e+04  4.643662e+04  0.000000e+00
0       carts         cpu         high              2 2024-01-19 16:57:00+00:00         7.929470e+01  7.914544e+01  1.413988e+00
6       carts    workload          low              2 2024-01-19 17:04:00+00:00         7.864440e+00  8.657317e+00  7.876905e+00
7   catalogue         cpu         high              2 2024-01-19 17:04:00+00:00         2.131040e-01  2.081030e-01  1.650370e-01
9   catalogue  latency-90         high              2 2024-01-19 16:59:00+00:00         6.448000e-03  6.282000e-03  4.627000e-03
8   catalogue  latency-50         high              2 2024-01-19 16:59:00+00:00         3.317000e-03  3.307000e-03  3.015000e-03
10  catalogue         mem         high              2 2024-01-19 17:06:00+00:00         6.114181e+06  6.109239e+06  6.005328e+06
12  catalogue    workload          low              2 2024-01-19 17:01:00+00:00         3.976700e+00  4.364614e+00  3.979380e+00
5       carts      socket         high              2 2024-01-19 17:01:00+00:00         1.586667e+01  1.547667e+01  9.600000e+00
2       carts  latency-50         high              2 2024-01-19 17:04:00+00:00         5.511200e-02  5.330100e-02  1.108600e-02
14     orders  latency-50         high              2 2024-01-19 17:01:00+00:00         7.131700e-02  7.036700e-02  1.838700e-02
13     orders         cpu         high              2 2024-01-19 16:54:00+00:00         2.455599e+00  2.217785e+00  1.034730e+00
15     orders  latency-90         high              2 2024-01-19 16:56:00+00:00         9.541300e-02  9.501600e-02  2.698100e-02
16     orders         mem         high              2 2024-01-19 16:54:00+00:00         3.359950e+08  3.358083e+08  3.343617e+08
19    payment         cpu         high              2 2024-01-19 17:00:00+00:00         9.762700e-02  9.631200e-02  7.857100e-02,      cmdb_id       log_name anomaly_type  anomaly_count      earliest_anomaly_utc  max_value_in_window  global_p95  global_p5
0      carts  log.row_count          low              8 2024-01-19 16:45:00+00:00                   25       100.0       25.0
1  catalogue  log.row_count          low              2 2024-01-19 16:43:00+00:00                  105       170.2      115.0
2     orders  log.row_count          low              2 2024-01-19 16:43:00+00:00                   84       130.0       89.6
3    payment  log.row_count          low              2 2024-01-19 16:43:00+00:00                   66       105.0       72.0
4       user  log.row_count          low              2 2024-01-19 16:43:00+00:00                  395       617.0      424.8)```
```